# 🤝 The Pivot — Co-Founder Advisor (Qwen2.5-3B)
**Meta PyTorch OpenEnv Hackathon 2026**

This is a **conversational** variant of the original training notebook.

### What's different from `train_colab.ipynb`?
| | Original (1.5B) | This notebook (3B Advisor) |
|---|---|---|
| Model | Qwen2.5-1.5B-Instruct | **Qwen2.5-3B-Instruct** |
| Output style | `DECISION: EXECUTE` | Natural advisor reply ("Honestly, I'd push forward...") |
| Memory | None | **Rolling 3-turn conversation memory** |
| Persona | Pivot expert | **Co-founder advisor (Alex, 15y experience)** |
| Episodes | 80 × 60 steps | 100 × 20 steps |
| Training time on T4 | ~2h | **~2.5–3h** |
| Reward | Env reward only | Env reward **+ conversation quality bonuses** |
| Learning grade | None | **Per-episode + live plot every 5 episodes** |

### Why 3B and not 5B?
- 3B at 4-bit ≈ 1.8GB weights + ref copy ≈ ~6GB total → fits T4 (15GB) with **room to spare**.
- 5B is borderline OOM with ref model + KL penalty on T4.
- 3B has 2x the reasoning capacity of 1.5B but trains in similar wall-clock time.

### How many episodes?
- **Minimum to see learning**: 60 episodes (~1.5h)
- **Sweet spot**: **100 episodes (~2.5–3h)** ← default in this notebook
- **Extended**: 150 episodes (~4–4.5h) — risky on free Colab (12h session)

Run every cell top to bottom. Runtime → Change runtime type → **T4 GPU** first.

In [ ]:
# Cell 1 — Check GPU + estimate time
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — go to Runtime → Change runtime type → T4')
print('CUDA available:', torch.cuda.is_available())
print()
print('⏱️  Estimated training time on T4 (Qwen2.5-3B, 100 episodes):')
print('     - Episode rollout (20 steps):     ~30-40s')
print('     - GRPO update (4 priority steps): ~15-25s')
print('     - Per episode total:              ~50-65s')
print('     - 100 episodes:                   ~2.5-3 hours')
print('     - 60 episodes (minimum):          ~1.5 hours')

In [ ]:
# Cell 2 — Install packages
%%capture
!pip install -q openenv-core "transformers>=4.45.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" wandb pydantic numpy python-dotenv
print('done')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_advisor_3b'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Saving to: {SAVE_DIR}')

In [ ]:
# Cell 4 — Clone project
import os, sys
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler
sys.path.insert(0, '/content/meta_scaler')
os.chdir('/content/meta_scaler')

try:
    from models import CoFounderAction, ActionType, CoFounderObservation
    from server.cofounder_environment import CoFounderEnvironment
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports OK!')
    print('Actions available:', len(list(ActionType)))
    print('Difficulty ladder:', DIFFICULTY_LADDER)
except ImportError as e:
    print(f'❌ Import error: {e}')

In [ ]:
# Cell 5 — W&B login
import wandb
wandb.login()   # paste your key from wandb.ai/settings
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

In [ ]:
# Cell 6 — Load Qwen2.5-3B + LoRA + gradient checkpointing
# 3B at 4-bit ≈ 1.8GB. With ref model copy ≈ ~5-6GB total. Plenty of room on T4 (15GB).
# Gradient checkpointing reduces activation memory at the cost of ~25% slower backprop.
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME       = 'Qwen/Qwen2.5-3B-Instruct'
DEVICE           = 'cuda'
MAX_SEQ_LEN      = 768   # longer than original (512) to fit conversation history
TRAIN_MAX_TOKENS = 70    # 2-3 sentence advisor reply (was 16 for action token)
DEMO_MAX_TOKENS  = 180   # full readable response in demo

print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map='auto',
    trust_remote_code=True,
)

# More target modules + slightly higher rank than original (1.5B used r=8 with 2 modules).
# 3B has more capacity — push LoRA harder.
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
))

model.gradient_checkpointing_enable()       # saves memory
model.enable_input_require_grads()           # gradients flow into LoRA through frozen 4-bit base

model.print_trainable_parameters()
gc.collect(); torch.cuda.empty_cache()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model ready!  GPU: {used:.2f} GB used / {total:.1f} GB total  ({total-used:.1f} GB free)')
print(f'   MAX_SEQ_LEN={MAX_SEQ_LEN}  TRAIN_MAX_TOKENS={TRAIN_MAX_TOKENS}  DEMO_MAX_TOKENS={DEMO_MAX_TOKENS}')

In [ ]:
# Cell 7 — Advisor persona + ConversationMemory + obs formatter
# This is the heart of the conversational variant. Three pieces:
#   1. ADVISOR_SYSTEM_PROMPT — role definition. NO action labels allowed.
#   2. ConversationMemory — rolling 3-turn history fed into every prompt.
#   3. obs_to_advisor_message — turns env observation into a natural co-founder update.

ADVISOR_SYSTEM_PROMPT = (
    "You are Alex, a no-nonsense co-founder advisor with 15+ years scaling startups. "
    "Your co-founder brings you the company's metrics each month. "
    "Reply like a real advisor would: 2-3 sentences, conversational tone, direct and specific. "
    "Reference the actual numbers when they matter. "
    "If you've talked before, reference what you said last time. "
    "NEVER use labels like 'Action:', 'Decision:', 'Recommendation:' — just speak naturally. "
    "You are NOT a pivot expert — you are a balanced co-founder who knows when to execute, "
    "when to research, when to fundraise, when to cut, and yes, sometimes when to pivot."
)


class ConversationMemory:
    """Rolling buffer of (user, assistant) turns. Resets per episode."""
    def __init__(self, max_turns: int = 3):
        self.history = []      # list of {'role': 'user'/'assistant', 'content': str}
        self.max_turns = max_turns

    def add_user(self, content: str):
        self.history.append({'role': 'user', 'content': content})

    def add_assistant(self, content: str):
        self.history.append({'role': 'assistant', 'content': content})

    def get_messages(self):
        recent = self.history[-(self.max_turns * 2):]   # last N user+assistant pairs
        return [{'role': 'system', 'content': ADVISOR_SYSTEM_PROMPT}] + recent

    def reset(self):
        self.history = []

    def last_assistant_words(self) -> set:
        for msg in reversed(self.history[:-1]):   # skip current user msg
            if msg['role'] == 'assistant':
                return set(msg['content'].lower().split())
        return set()


def obs_to_advisor_message(obs) -> str:
    """Turn an env observation into a natural co-founder check-in message."""
    runway   = getattr(obs, 'runway_remaining', 0) or 0
    revenue  = getattr(obs, 'monthly_revenue', 0) or 0
    burn     = getattr(obs, 'monthly_burn', 0) or 0
    churn    = getattr(obs, 'churn_rate', 0) or 0
    nps      = getattr(obs, 'nps_score', 0) or 0
    pmf      = getattr(obs, 'product_market_fit', 0) or 0
    morale   = getattr(obs, 'team_morale', 0) or 0
    comp     = getattr(obs, 'competitor_strength', 0) or 0
    step     = getattr(obs, 'step', 0) or 0

    return (
        f"Month {step} update:\n"
        f"• Runway: {runway}mo  |  Burn: ${burn:,.0f}/mo\n"
        f"• Revenue: ${revenue:,.0f}/mo  |  Churn: {churn:.0%}  |  NPS: {nps:.0f}\n"
        f"• PMF: {pmf:.2f}/1.0  |  Team morale: {morale:.2f}/1.0\n"
        f"• Competitor strength: {comp:.2f}/1.0\n\n"
        f"What's your call?"
    )


# Quick smoke test
from server.cofounder_environment import CoFounderEnvironment
_env = CoFounderEnvironment(); _obs = _env.reset()
_mem = ConversationMemory()
_mem.add_user(obs_to_advisor_message(_obs))
print('Sample user message to advisor:')
print('-' * 60)
print(_mem.history[-1]['content'])
print('-' * 60)
print(f'✅ Advisor persona + memory ready ({_mem.max_turns}-turn rolling buffer)')

In [ ]:
# Cell 8 — Action extractor (from natural advisor text) + advisor phrase bank
# CHALLENGE: the model speaks naturally ("let's keep building") but the env needs an ActionType.
# SOLUTION: keyword-based extractor + advisor phrase bank for ε-greedy exploration.
import re, random
from models import ActionType

# Keyword → ActionType map. Order matters — more specific phrases checked first.
ADVICE_KEYWORDS = [
    # PIVOT — must come before generic "change"
    (['pivot', 'change direction', 'new market', 'rethink the direction', 'different angle', 'wrong market'], ActionType.PIVOT),
    # FIRE — must come before HIRE
    (['fire ', 'let someone go', 'let them go', 'part ways', 'underperformer', 'cut the underperformer'], ActionType.FIRE),
    # HIRE
    (['hire', 'bring on', 'add to the team', 'new engineer', 'make a key hire'], ActionType.HIRE),
    # CUT_COSTS
    (['cut costs', 'cut cost', 'reduce burn', 'tighten the belt', 'reduce spend', 'trim'], ActionType.CUT_COSTS),
    # FUNDRAISE
    (['fundrais', 'raise money', 'raise capital', 'raise a round', 'pitch investors', 'start a round', 'go to investors'], ActionType.FUNDRAISE),
    # SELL
    (['sell the company', 'acquisition', 'acqui-hire', 'strategic exit', 'find an acquirer', 'potential buyers'], ActionType.SELL),
    # LAUNCH_FEATURE
    (['ship the feature', 'launch the feature', 'release the feature', 'ship it', 'get the feature out', 'launch it'], ActionType.LAUNCH_FEATURE),
    # MARKETING_CAMPAIGN
    (['marketing campaign', 'run a campaign', 'marketing push', 'invest in marketing', 'top-of-funnel', 'push on growth'], ActionType.MARKETING_CAMPAIGN),
    # SET_PRICING
    (['pricing', 'price point', 'raise prices', 'price adjustment', 'pricing model'], ActionType.SET_PRICING),
    # PARTNERSHIP
    (['partnership', 'partner up', 'strategic partner', 'partner with', 'partner deal'], ActionType.PARTNERSHIP),
    # RESEARCH
    (['talk to customers', 'customer discovery', 'research', 'validate', 'find out why', 'more data'], ActionType.RESEARCH),
    # EXECUTE — fallback / default
    (['keep pushing', 'execute', 'stay the course', 'keep building', 'focus on execution', 'on the right track'], ActionType.EXECUTE),
]


def extract_action(text: str) -> ActionType:
    """Map natural advisor text → ActionType. Default to EXECUTE if no clear signal."""
    t = text.lower()
    for keywords, action in ADVICE_KEYWORDS:
        for kw in keywords:
            if kw in t:
                return action
    return ActionType.EXECUTE   # safe default


# Phrase bank for ε-greedy exploration. When we pick a random action,
# we pair it with a realistic advisor phrase — so training data is ALWAYS natural language.
ADVISOR_PHRASES = {
    ActionType.EXECUTE: [
        "Keep pushing forward with the current plan. The metrics are stable enough to justify staying the course.",
        "Honestly, I'd just keep building. We're on the right track and don't need to change anything yet.",
        "Stay the course and focus on execution. Don't get distracted by shiny new ideas this month.",
    ],
    ActionType.PIVOT: [
        "Honestly, I think it's time to rethink the direction. The numbers aren't pointing us toward product-market fit here.",
        "We need to pivot. This market isn't responding the way we hoped and the runway won't last forever.",
        "The data is telling us the current angle isn't working. Let's talk about a real pivot.",
    ],
    ActionType.RESEARCH: [
        "Before we do anything else, go talk to ten customers this week. The churn is a signal we don't fully understand yet.",
        "Let's validate before we build anything new. Run customer discovery and find out what's actually broken.",
        "We need more data. Spend two weeks on research before committing to a direction.",
    ],
    ActionType.FUNDRAISE: [
        "Time to raise. Get in front of investors now while you still have leverage and the metrics look good.",
        "Start a round this month. Don't wait until the runway gets critical — fundraising takes longer than you think.",
        "Hit up investors. The story is good enough to pitch right now.",
    ],
    ActionType.HIRE: [
        "We need to hire. The team is stretched and one good hire will unblock the next phase of growth.",
        "Make a key hire this month. Morale is fine and we have the budget to bring someone in.",
        "Bring someone on to absorb the load. We're losing velocity because the team is too thin.",
    ],
    ActionType.CUT_COSTS: [
        "We need to tighten the belt. Cut every non-essential expense this month — every dollar buys runway.",
        "Reduce the burn rate now. The runway is short enough that we can't afford anything that isn't moving the needle.",
        "Time to trim. Identify what's not driving revenue and cut it immediately.",
    ],
    ActionType.SELL: [
        "We should explore a strategic acquisition. The timing is right and it's worth knowing your options.",
        "Let's start conversations with potential acquirers. An exit at this stage might make sense.",
        "Reach out to potential buyers quietly. It's smart to know what's on the table.",
    ],
    ActionType.LAUNCH_FEATURE: [
        "Ship the feature this week. We've been sitting on it too long and the team needs the win.",
        "Get the feature out the door. More shipping equals more feedback equals faster iteration.",
        "Launch it. Users are waiting and every week of delay shows up in the churn numbers.",
    ],
    ActionType.MARKETING_CAMPAIGN: [
        "Run a marketing campaign this month. The product is solid — we just need top-of-funnel attention.",
        "Time to invest in marketing. The unit economics justify the spend and we need to push on growth.",
        "Launch a campaign. PMF is good enough that paid acquisition will pay back.",
    ],
    ActionType.SET_PRICING: [
        "Revisit the pricing this month. There's room to optimize and it directly hits the LTV:CAC ratio.",
        "Test a new price point. The current pricing is leaving money on the table.",
        "Update the pricing model. Align it with the actual value we're delivering to customers.",
    ],
    ActionType.FIRE: [
        "We need to make a hard call and let someone go. It's painful but it protects the rest of the team.",
        "Cut the underperformer this month. Morale is suffering because the rest of the team sees it.",
        "This is tough, but we need to part ways with someone. It's the right call for the company.",
    ],
    ActionType.PARTNERSHIP: [
        "Explore a strategic partnership. Distribution is the bottleneck — not the product.",
        "Find a partner who already has the users we need. It's faster than building our own funnel.",
        "Look for a partnership deal this month. The right one unlocks the next stage of growth.",
    ],
}

# Weighted exploration pool (mirrors original notebook's distribution)
RANDOM_ACTIONS = (
    [ActionType.EXECUTE]            * 5 +
    [ActionType.RESEARCH]           * 3 +
    [ActionType.LAUNCH_FEATURE]     * 3 +
    [ActionType.FUNDRAISE]          * 2 +
    [ActionType.HIRE]               * 1 +
    [ActionType.CUT_COSTS]          * 1 +
    [ActionType.MARKETING_CAMPAIGN] * 1 +
    [ActionType.PARTNERSHIP]        * 1 +
    [ActionType.PIVOT]              * 1
)

def random_advisor_response():
    """Pick a random action and pair it with a natural advisor phrase."""
    action = random.choice(RANDOM_ACTIONS)
    phrase = random.choice(ADVISOR_PHRASES[action])
    return phrase, action


# Quick smoke test
print('Action extractor smoke test:')
for sample in [
    "Honestly, I'd just keep building this month.",
    "We need to pivot — this market is dead.",
    "Talk to ten customers before we do anything else.",
    "Time to raise. Get in front of investors.",
    "Cut the underperformer. Painful but right.",
]:
    print(f'  {sample[:55]:55s} → {extract_action(sample).value}')
print(f'\n✅ Extractor + phrase bank ready ({sum(len(v) for v in ADVISOR_PHRASES.values())} phrases)')

In [ ]:
# Cell 9 — Conversation reward + LearningTracker (per-episode learning grade)
# Reward = base env reward + conversation quality bonuses + tone penalties.
# This is what teaches the model to TALK like an advisor, not just pick correct actions.
import numpy as np

BAD_LABEL_PATTERNS = ['decision:', 'action:', 'recommendation:', 'choice:', 'answer:']
GOOD_OPENERS = ('i think', 'we should', 'honestly', 'let\'s', 'lets', 'right now',
                'given', 'at this point', 'time to', 'the numbers', 'your', 'time')


def compute_conversation_reward(response: str, base_env_reward: float, obs, prev_assistant_words: set) -> float:
    """
    Layered reward:
      1. base_env_reward / 100   → normalized environment signal (-6 to +2 range)
      2. metric_awareness +0.3   → mentions actual numbers from obs
      3. memory_bonus    +0.2    → references prior conversation (word overlap)
      4. length_bonus    ±0.3   → advisor-quality length (20-80 words ideal)
      5. label_penalty   -0.5   → starts with 'DECISION:' / 'Action:' (we banned this)
      6. tone_bonus      +0.1   → starts with natural advisor phrasing
    """
    reward = base_env_reward / 100.0   # normalize to roughly [-6, +2]
    text   = response.lower().strip()
    words  = text.split()
    n      = len(words)

    # 2. Metric awareness
    runway  = int(getattr(obs, 'runway_remaining', 0) or 0)
    revenue = int(getattr(obs, 'monthly_revenue', 0) or 0)
    nps     = int(getattr(obs, 'nps_score', 0) or 0)
    metric_strs = {str(runway), str(revenue), str(nps)}
    if any(m in text and m != '0' for m in metric_strs):
        reward += 0.3
    elif any(kw in text for kw in ('runway', 'revenue', 'churn', 'nps', 'morale', 'pmf', 'burn')):
        reward += 0.15   # half-credit for mentioning category, not number

    # 3. Memory / conversation continuity bonus
    if prev_assistant_words and n > 0:
        overlap = len(set(words) & prev_assistant_words) / n
        if overlap > 0.10:           # at least 10% word overlap with prior turn
            reward += 0.2

    # 4. Length quality
    if 20 <= n <= 80:
        reward += 0.1
    elif n < 8:
        reward -= 0.3                # too terse
    elif n > 150:
        reward -= 0.2                # rambling

    # 5. Label penalty — we want NATURAL responses
    head = text[:40]
    if any(p in head for p in BAD_LABEL_PATTERNS):
        reward -= 0.5

    # 6. Tone bonus
    if text.startswith(GOOD_OPENERS):
        reward += 0.1

    return reward


class LearningTracker:
    """Records per-episode signals so we can grade learning live."""
    def __init__(self):
        self.rewards          = []   # total per-episode reward (with bonuses)
        self.env_rewards      = []   # raw env reward only (apples-to-apples vs original notebook)
        self.response_lengths = []   # avg words per advisor response
        self.action_diversity = []   # unique actions used per episode
        self.memory_bonus_rate= []   # % of turns that referenced prior conversation
        self.label_violations = []   # % of turns that started with banned label (lower = better)

    def record(self, steps: list):
        if not steps:
            return
        self.rewards.append(sum(s['reward'] for s in steps))
        self.env_rewards.append(sum(s['base_reward'] for s in steps))
        self.response_lengths.append(np.mean([len(s['completion'].split()) for s in steps]))
        self.action_diversity.append(len(set(s['action'] for s in steps)))
        self.memory_bonus_rate.append(np.mean([s.get('used_memory', 0) for s in steps]))
        self.label_violations.append(np.mean([s.get('label_violation', 0) for s in steps]))

    def grade(self) -> str:
        """Return an A/B/C/D grade based on recent learning trend."""
        if len(self.rewards) < 10:
            return f'⏳ N/A (need 10 episodes, have {len(self.rewards)})'
        recent = self.rewards[-10:]
        prev   = self.rewards[-20:-10] if len(self.rewards) >= 20 else self.rewards[:10]
        delta  = float(np.mean(recent) - np.mean(prev))
        avg_len = float(np.mean(self.response_lengths[-10:]))
        violations = float(np.mean(self.label_violations[-10:]))
        memory_use = float(np.mean(self.memory_bonus_rate[-10:]))

        if delta > 0.5 and violations < 0.1 and 15 <= avg_len <= 90:
            grade = '🟢 A — Learning well'
        elif delta > 0.0 and violations < 0.2:
            grade = '🟡 B — Slow improvement'
        elif delta > -0.5:
            grade = '🟠 C — Plateau'
        else:
            grade = '🔴 D — Regressing'
        return (f'{grade}  |  rewardΔ {delta:+.2f}  |  '
                f'avg_len {avg_len:.0f}w  |  memory_use {memory_use:.0%}  |  '
                f'label_violations {violations:.0%}')


print('✅ Reward function + LearningTracker ready')

In [ ]:
# Cell 10 — GRPO helpers (adapted for conversational completions)
import torch, gc
from models import CoFounderAction
from server.cofounder_environment import CoFounderEnvironment


@torch.no_grad()
def generate_advisor_response(messages: list, max_tokens: int = None):
    """Generate a natural-language advisor response. Returns (text, ActionType)."""
    if max_tokens is None:
        max_tokens = TRAIN_MAX_TOKENS
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    out = model.generate(
        **inp, max_new_tokens=max_tokens,
        do_sample=True, temperature=0.85, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    del inp, out
    return decoded, extract_action(decoded)


def get_log_prob(prompt: str, completion: str) -> torch.Tensor:
    """Mean log probability of `completion` tokens given `prompt`. Differentiable."""
    comp_ids = tokenizer(completion, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
    if comp_ids.shape[0] == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)
    prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]
    full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
    p_len = prompt_ids.shape[0]
    out = model(input_ids=full_ids)
    comp_logits = out.logits[0, p_len - 1:-1, :].clone()
    target_ids  = full_ids[0, p_len:].clone()
    del out, full_ids
    torch.cuda.empty_cache()
    lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
    return lp[range(len(target_ids)), target_ids].mean()


MAX_STEPS_PER_EPISODE = 20   # shorter than original (60) — conversational training is heavier per step


def run_conversational_episode(scenario: dict, seed: int = 0, epsilon: float = 0.5) -> list[dict]:
    """Run one conversational episode with rolling memory."""
    import random
    env = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
    obs = env.reset()
    memory = ConversationMemory(max_turns=3)
    steps = []

    for _ in range(MAX_STEPS_PER_EPISODE):
        # 1. Format the user message from observation
        user_msg = obs_to_advisor_message(obs)
        memory.add_user(user_msg)
        prev_assistant_words = memory.last_assistant_words()

        # 2. Build prompt from rolling memory
        messages = memory.get_messages()
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # 3. Generate response (ε-greedy exploration with NATURAL phrases)
        if random.random() < epsilon:
            completion, action_type = random_advisor_response()
        else:
            completion, action_type = generate_advisor_response(messages)

        memory.add_assistant(completion)

        # 4. Step environment
        next_obs = env.step(CoFounderAction(action_type=action_type))
        base_reward = float(next_obs.reward or 0)

        # 5. Compute conversational reward (env + bonuses)
        conv_reward = compute_conversation_reward(completion, base_reward, obs, prev_assistant_words)

        # Track signals for LearningTracker
        used_memory = 1 if (prev_assistant_words and
                            len(set(completion.lower().split()) & prev_assistant_words) / max(len(completion.split()), 1) > 0.10
                           ) else 0
        label_violation = 1 if any(p in completion.lower()[:40] for p in BAD_LABEL_PATTERNS) else 0

        steps.append({
            'prompt':          prompt,
            'completion':      completion,
            'action':          action_type.value,
            'reward':          conv_reward,
            'base_reward':     base_reward,
            'step':            next_obs.step,
            'runway':          next_obs.runway_remaining,
            'used_memory':     used_memory,
            'label_violation': label_violation,
        })
        obs = next_obs
        if obs.done:
            break
    return steps


# Sanity check: gradient flow through 3B model with conversational completion
model.train()
_env = CoFounderEnvironment(); _obs = _env.reset()
_mem = ConversationMemory(); _mem.add_user(obs_to_advisor_message(_obs))
_prompt = tokenizer.apply_chat_template(_mem.get_messages(), tokenize=False, add_generation_prompt=True)
_lp = get_log_prob(_prompt, "Honestly, let's keep building this month. The metrics are stable.")
print(f'Gradient check:  log_prob={_lp.item():.4f}  requires_grad={_lp.requires_grad}')
assert _lp.requires_grad and _lp.grad_fn is not None, '❌ Gradients not flowing!'
model.eval()
print(f'✅ Helpers ready  |  MAX_STEPS_PER_EPISODE={MAX_STEPS_PER_EPISODE}')

In [ ]:
# Cell 11 — Config + optimizer + ref model + curriculum
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
import copy

CONFIG = {
    'n_episodes':        100,    # ~2.5-3h on T4 (sweet spot)
    'lr':                3e-5,   # slightly lower than 1.5B run — 3B is more sensitive
    'grad_clip':         0.5,
    'grpo_steps_sample': 4,      # fewer than original (8) — long completions are expensive
    'save_every':        20,
    'kl_beta':           0.02,
    'adv_clip':          1.5,
    'reward_scale':      1.0,    # rewards already normalized in compute_conversation_reward
}

optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)

# Reference model for KL penalty
print('Cloning reference model (frozen) for KL penalty…')
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

# Running reward baseline
reward_baseline = 0.0
BASELINE_DECAY  = 0.95


def get_log_prob_nograd(prompt: str, completion: str) -> float:
    with torch.no_grad():
        comp_ids = tokenizer(completion, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        if comp_ids.shape[0] == 0:
            return 0.0
        prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True,
                               max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]
        full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
        p_len = prompt_ids.shape[0]
        out = ref_model(input_ids=full_ids)
        comp_logits = out.logits[0, p_len - 1:-1, :]
        target_ids = full_ids[0, p_len:]
        lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
        val = lp[range(len(target_ids)), target_ids].mean().item()
        del out, full_ids; torch.cuda.empty_cache()
        return val


def grpo_update(episode_steps: list, ep_reward: float) -> dict:
    global reward_baseline
    import random
    model.train()
    optimizer.zero_grad()

    rewards = np.array([s['reward'] / CONFIG['reward_scale'] for s in episode_steps], dtype=np.float32)
    baseline_norm = reward_baseline / CONFIG['reward_scale']
    adv = rewards - baseline_norm
    std = adv.std()
    if std > 1e-6:
        adv = adv / std
    adv = np.clip(adv, -CONFIG['adv_clip'], CONFIG['adv_clip'])

    reward_baseline = BASELINE_DECAY * reward_baseline + (1 - BASELINE_DECAY) * ep_reward

    sample_k = min(CONFIG['grpo_steps_sample'], len(episode_steps))
    abs_adv = np.abs(adv)
    probs   = abs_adv / (abs_adv.sum() + 1e-8)
    if probs.sum() < 0.01:
        indices = random.sample(range(len(episode_steps)), sample_k)
    else:
        indices = np.random.choice(len(episode_steps), size=sample_k, replace=False, p=probs).tolist()

    total_loss = 0.0; total_kl = 0.0; did_bwd = False
    for idx in indices:
        sd  = episode_steps[idx]
        lp  = get_log_prob(sd['prompt'], sd['completion'])
        ref = get_log_prob_nograd(sd['prompt'], sd['completion'])

        grpo_loss = (-lp * float(adv[idx])) / sample_k
        kl        = (lp - ref) / sample_k
        loss      = grpo_loss + CONFIG['kl_beta'] * kl

        if loss.requires_grad and loss.grad_fn is not None:
            loss.backward(); did_bwd = True
        total_loss += grpo_loss.item(); total_kl += kl.item()
        del lp, kl, loss
        gc.collect(); torch.cuda.empty_cache()

    if did_bwd:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()
    optimizer.zero_grad()
    model.eval()
    gc.collect(); torch.cuda.empty_cache()
    return {'loss': total_loss, 'kl': total_kl / sample_k}


print('✅ Config + optimizer + ref model + curriculum ready')
print(f'   n_episodes={CONFIG["n_episodes"]}  lr={CONFIG["lr"]}  grpo_sample_k={CONFIG["grpo_steps_sample"]}')
print(f'   max_steps_per_episode={MAX_STEPS_PER_EPISODE}')

In [ ]:
# Cell 12 — TRAIN with live incremental learning display
import os, time
import matplotlib.pyplot as plt
from IPython.display import clear_output

os.environ.pop('WANDB_MODE', None)
run = wandb.init(
    project=WANDB_PROJECT,
    name='advisor-qwen3b-grpo',
    config={**CONFIG, 'model': MODEL_NAME, 'max_steps_per_episode': MAX_STEPS_PER_EPISODE},
    tags=['grpo', 'qwen3b', 'advisor', 'conversational', 'hackathon'],
)
print(f'W&B: {run.url}')
print('-' * 90)

epsilon   = 0.55
EPS_DECAY = 0.985
EPS_MIN   = 0.15

tracker          = LearningTracker()
episode_lengths  = []
all_losses       = []
best_reward      = float('-inf')
start_time       = time.time()
model.eval()


def live_progress_plot(tracker, ep, total_eps):
    """Refresh inline plot every 5 episodes so the user sees learning live."""
    if ep % 5 != 0 and ep != total_eps:
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    # Reward curve
    axes[0].plot(tracker.rewards, alpha=0.35, color='tab:blue', label='per-episode')
    if len(tracker.rewards) >= 5:
        smooth = np.convolve(tracker.rewards, np.ones(5)/5, mode='valid')
        axes[0].plot(range(4, len(tracker.rewards)), smooth, color='tab:blue', lw=2, label='5-ep avg')
    axes[0].axhline(0, color='gray', ls='--', alpha=0.5)
    axes[0].set_title('Total Reward (with conversation bonuses)')
    axes[0].set_xlabel('Episode'); axes[0].legend(); axes[0].grid(alpha=0.3)
    # Response length
    axes[1].plot(tracker.response_lengths, color='tab:green', alpha=0.6)
    axes[1].axhline(40, color='orange', ls='--', label='target ~40 words')
    axes[1].set_title('Avg Advisor Response Length (words)')
    axes[1].set_xlabel('Episode'); axes[1].legend(); axes[1].grid(alpha=0.3)
    # Memory use rate
    axes[2].plot(tracker.memory_bonus_rate, color='tab:purple', alpha=0.6, label='memory use')
    axes[2].plot(tracker.label_violations, color='tab:red', alpha=0.6, label='label violations')
    axes[2].set_ylim(0, 1)
    axes[2].set_title('Conversation Quality Signals')
    axes[2].set_xlabel('Episode'); axes[2].legend(); axes[2].grid(alpha=0.3)
    fig.suptitle(f'Episode {ep}/{total_eps}   |   {tracker.grade()}', fontsize=11)
    plt.tight_layout()
    plt.show()


for ep in range(1, CONFIG['n_episodes'] + 1):
    scenario      = curriculum.sample_scenario()
    scenario_name = scenario.get('name', 'default')
    seed = ep * 13 + 7

    steps = run_conversational_episode(scenario, seed=seed, epsilon=epsilon)
    tracker.record(steps)

    ep_reward     = sum(s['reward'] for s in steps)
    ep_env_reward = sum(s['base_reward'] for s in steps)
    ep_length     = len(steps)
    survived      = ep_length >= MAX_STEPS_PER_EPISODE
    pivot_count   = sum(1 for s in steps if s['action'] == 'PIVOT')
    unique_acts   = len(set(s['action'] for s in steps))
    avg_resp_len  = float(np.mean([len(s['completion'].split()) for s in steps]))

    episode_lengths.append(ep_length)
    if ep_reward > best_reward:
        best_reward = ep_reward

    loss_stats = grpo_update(steps, ep_reward)
    all_losses.append(loss_stats['loss'])
    curriculum.record_result(ep_env_reward, survived)
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    gpu_gb  = torch.cuda.memory_allocated() / 1e9
    elapsed = (time.time() - start_time) / 60
    eta_min = (elapsed / ep) * (CONFIG['n_episodes'] - ep)
    mean10  = float(np.mean(tracker.rewards[-10:])) if len(tracker.rewards) >= 10 else float(np.mean(tracker.rewards))

    print(
        f'Ep {ep:3d}/{CONFIG["n_episodes"]} | {scenario_name:14s} | T{curriculum.current_tier+1} | '
        f'r {ep_reward:+6.2f} | env {ep_env_reward:+7.1f} | mean10 {mean10:+6.2f} | '
        f'len {avg_resp_len:4.0f}w | acts {unique_acts:2d} | piv {pivot_count} | '
        f'ε={epsilon:.2f} | loss {loss_stats["loss"]:+7.3f} | '
        f'GPU {gpu_gb:.1f}GB | {elapsed:.0f}m elapsed, ~{eta_min:.0f}m left'
    )

    wandb.log({
        'episode':              ep,
        'ep_reward_total':      ep_reward,
        'ep_reward_env':        ep_env_reward,
        'best_reward':          best_reward,
        'mean_reward_10ep':     mean10,
        'ep_length':            ep_length,
        'avg_response_length':  avg_resp_len,
        'unique_actions':       unique_acts,
        'pivot_count':          pivot_count,
        'memory_use_rate':      tracker.memory_bonus_rate[-1],
        'label_violation_rate': tracker.label_violations[-1],
        'loss':                 loss_stats['loss'],
        'kl':                   loss_stats.get('kl', 0),
        'epsilon':              epsilon,
        'curriculum_tier':      curriculum.current_tier + 1,
        'gpu_gb':               gpu_gb,
        'scenario':             scenario_name,
    })

    if curriculum.should_advance() and curriculum.advance_tier():
        new_scenario = DIFFICULTY_LADDER[curriculum.current_tier]
        print(f'\n   → CURRICULUM ADVANCE → Tier {curriculum.current_tier+1}/5: {new_scenario}\n')
        wandb.log({'curriculum_advance': curriculum.current_tier, 'episode': ep})

    # Live visual feedback every 5 episodes
    if ep % 5 == 0 or ep == CONFIG['n_episodes']:
        live_progress_plot(tracker, ep, CONFIG['n_episodes'])

    # Checkpoint
    if ep % CONFIG['save_every'] == 0:
        ckpt = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        print(f'   💾 Checkpoint → {ckpt}')

print('\n' + '=' * 90)
print('TRAINING COMPLETE')
print(f'  Episodes              : {CONFIG["n_episodes"]}')
print(f'  Wall time             : {(time.time()-start_time)/60:.1f} min')
print(f'  Best reward           : {best_reward:.2f}')
print(f'  Mean reward last 10ep : {np.mean(tracker.rewards[-10:]):.2f}')
print(f'  Final tier            : {curriculum.current_tier+1}/5')
print(f'  Final grade           : {tracker.grade()}')
print('=' * 90)

In [ ]:
# Cell 13 — Save final model + close W&B
FINAL = f'{SAVE_DIR}/final_advisor_3b'
model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
print(f'✅ Final advisor model saved to {FINAL}')
wandb.finish()
print('✅ W&B closed')

In [ ]:
# Cell 14 — Live conversation test
# Talks to the advisor through a real episode and shows the natural dialogue.
# This is what the trained model should sound like — a co-founder, not an action picker.
from training.curriculum import AdaptiveCurriculum

def live_conversation_demo(scenario_name='b2c_saas', seed=999, max_steps=12):
    c = AdaptiveCurriculum()
    scenario = c._all_scenarios.get(scenario_name)
    if not scenario:
        print(f'❌ Scenario "{scenario_name}" not found'); return

    env = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
    obs = env.reset()
    memory = ConversationMemory(max_turns=3)
    model.eval()

    print(f'\n{"="*78}')
    print(f'💬 LIVE ADVISOR CONVERSATION   |   Scenario: {scenario_name}   |   Seed: {seed}')
    print(f'{"="*78}\n')

    for step in range(max_steps):
        user_msg = obs_to_advisor_message(obs)
        memory.add_user(user_msg)
        messages = memory.get_messages()

        # Use longer token budget + greedy for cleaner demo output
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inp, max_new_tokens=DEMO_MAX_TOKENS,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        del inp, out; torch.cuda.empty_cache()

        action_type = extract_action(response)
        memory.add_assistant(response)
        next_obs = env.step(CoFounderAction(action_type=action_type))

        print(f'🔹 You (Month {step+1}):')
        for line in user_msg.split('\n'):
            print(f'   {line}')
        print(f'\n🔸 Alex (advisor):')
        for line in response.split('\n'):
            print(f'   {line}')
        print(f'\n   → (extracted action: {action_type.value}, env reward: {next_obs.reward or 0:+.1f})')
        print(f'{"-"*78}')

        obs = next_obs
        if obs.done:
            print(f'\n⚠️  Episode ended early (runway={obs.runway_remaining})')
            break

    print(f'\n✅ Demo complete — the advisor should speak naturally, reference numbers, and recall prior turns.\n')


live_conversation_demo('b2c_saas', seed=999, max_steps=8)
live_conversation_demo('fintech',  seed=999, max_steps=8)

In [ ]:
# Cell 15 — Save plots + metrics
import matplotlib.pyplot as plt
import numpy as np, json, pathlib

PLOTS_DIR   = pathlib.Path('/content/meta_scaler/docs/plots/advisor_3b')
DRIVE_PLOTS = pathlib.Path(f'{SAVE_DIR}/plots/advisor_3b')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

def _smooth(xs, k=10):
    if len(xs) < k: return xs
    return np.convolve(xs, np.ones(k)/k, mode='valid')

# Reward curve
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(tracker.rewards, alpha=0.3, color='tab:blue', label='per-episode (with bonuses)')
ax.plot([r/100 for r in tracker.env_rewards], alpha=0.3, color='tab:gray', label='env reward / 100')
if len(tracker.rewards) >= 10:
    ax.plot(range(9, len(tracker.rewards)), _smooth(tracker.rewards, 10),
            color='tab:blue', linewidth=2, label='10-ep moving avg')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title('Qwen2.5-3B Advisor — Reward Curve (env + conversation bonuses)')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
for p in [PLOTS_DIR / 'reward_curve.png', DRIVE_PLOTS / 'reward_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight'); print(f'  saved → {p}')
plt.close(fig)

# Conversation quality curve
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(tracker.memory_bonus_rate, color='tab:purple', alpha=0.7, label='memory use rate')
ax.plot(tracker.label_violations, color='tab:red', alpha=0.7, label='label violation rate (lower better)')
ax.set_ylim(0, 1); ax.set_xlabel('Episode'); ax.set_ylabel('Rate')
ax.set_title('Conversation Quality Signals')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
for p in [PLOTS_DIR / 'conversation_quality.png', DRIVE_PLOTS / 'conversation_quality.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight'); print(f'  saved → {p}')
plt.close(fig)

# Loss curve
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(all_losses, alpha=0.3, color='tab:red', label='per-episode loss')
if len(all_losses) >= 10:
    ax.plot(range(9, len(all_losses)), _smooth(all_losses, 10),
            color='tab:red', linewidth=2, label='10-ep moving avg')
ax.set_xlabel('Episode'); ax.set_ylabel('GRPO loss')
ax.set_title('Qwen2.5-3B Advisor — Loss Curve')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
for p in [PLOTS_DIR / 'loss_curve.png', DRIVE_PLOTS / 'loss_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight'); print(f'  saved → {p}')
plt.close(fig)

# Metrics
metrics = {
    'variant':              'advisor_3b',
    'model':                MODEL_NAME,
    'n_episodes':           len(tracker.rewards),
    'max_steps_per_episode':MAX_STEPS_PER_EPISODE,
    'mean_reward_last_10':  float(np.mean(tracker.rewards[-10:])),
    'mean_env_reward_last_10': float(np.mean(tracker.env_rewards[-10:])),
    'best_reward':          float(max(tracker.rewards)),
    'avg_response_length':  float(np.mean(tracker.response_lengths[-10:])),
    'memory_use_rate':      float(np.mean(tracker.memory_bonus_rate[-10:])),
    'label_violation_rate': float(np.mean(tracker.label_violations[-10:])),
    'final_tier':           curriculum.current_tier + 1,
    'final_grade':          tracker.grade(),
}
for p in [PLOTS_DIR / 'metrics.json', DRIVE_PLOTS / 'metrics.json']:
    with open(p, 'w') as f: json.dump(metrics, f, indent=2)
print('\nMetrics:'); print(json.dumps(metrics, indent=2))
print(f'\nGit push: cd /content/meta_scaler && git add docs/plots/advisor_3b && git commit -m "Add 3B advisor plots" && git push')

In [ ]:
# Cell 16 — (Optional) Push trained advisor LoRA to HF Hub
from huggingface_hub import HfApi

HF_USERNAME  = 'Harshit-Makraria'
HF_REPO_NAME = 'the-pivot-advisor-3b'
HF_MODEL_ID  = f'{HF_USERNAME}/{HF_REPO_NAME}'

HF_TOKEN = input('Paste your HF token (hf_...): ').strip()
print(f'Pushing to {HF_MODEL_ID} …')

model.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)

card = f"""---
base_model: Qwen/Qwen2.5-3B-Instruct
tags:
  - peft
  - lora
  - rl
  - grpo
  - startup
  - advisor
  - conversational
  - hackathon
license: mit
---

# The Pivot — Co-Founder Advisor (Qwen2.5-3B, LoRA)

Conversational co-founder advisor fine-tuned with **GRPO** on the
[The Pivot](https://github.com/Harshit-Makraria/meta_scaler) OpenEnv environment.

- **Persona**: Alex, a no-nonsense co-founder advisor with 15+ years of startup experience.
- **Output style**: 2-3 sentence natural-language replies (not action labels).
- **Memory**: 3-turn rolling conversation memory.
- **Trained on**: 100 episodes × 20 steps across 5 startup scenarios.

## Usage
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
model = PeftModel.from_pretrained(base, '{HF_MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained('{HF_MODEL_ID}')
```

Built for the **Meta PyTorch OpenEnv Hackathon 2026**.
"""
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo='README.md',
    repo_id=HF_MODEL_ID,
    repo_type='model',
)
print(f'\n✅ Live at: https://huggingface.co/{HF_MODEL_ID}')